# MEMC cvičení 6 – Zamítací metoda

---

Inverzní transformace (cvičení 4) potřebuje distribuční funkci, kterou umíme invertovat. Co když to nejde? Normální rozdělení je typický příklad: $\Phi^{-1}$ nemá vzorec a numerická inverze je pomalá.

**Zamítací metoda** (anglicky *rejection method*) generuje z hustoty $f$, aniž bychom ji integrovali nebo invertovali. Potřebuje jen:

- umět **spočítat** $f(x)$ (stačí až na násobek, tedy bez normalizace),
- znát jednodušší hustotu $g$, ze které generovat umíme a jejíž násobek $M g$ leží **nad** $f$.

Princip: navrhneme hodnotu z jednoduchého rozdělení a pak ji s vhodnou pravděpodobností **zamítneme**, aby přijaté hodnoty měly přesně hustotu $f$.

V sešitu uvidíte:

- verzi s obdélníkem (nejjednodušší) a obrázek, proč funguje
- verzi s obálkou a **normální rozdělení z Cauchyho obálky**
- očekávaný počet pokusů a jak ovlivňuje volbu obálky
- co se stane, když je konstanta $M$ špatně, a jak to poznat

In [ ]:
import math
import random
import time

import matplotlib.pyplot as plt

random.seed(1)  # opakovatelné výsledky


def estimate(sampler, n):
    '''Běžné Monte Carlo: průměr n volání generátoru a jeho směrodatná chyba sigma/sqrt(n).'''
    s = s2 = 0.0
    for _ in range(n):
        x = sampler()
        s += x
        s2 += x * x
    mean = s / n
    var = max(s2 / n - mean * mean, 0.0)
    return mean, math.sqrt(var / n)


def check(name, mean, se, exact, k=5):
    '''Vypíše odhad a ověří, že leží do k směrodatných chyb od přesné hodnoty.'''
    print(f"{name}: {mean:.5f} +- {se:.5f}  (přesně {exact:.5f})")
    assert abs(mean - exact) <= k * se, name


# generátory z předchozích cvičení
def uniform(a, b):
    return a + (b - a) * random.random()


def exponential(lam):
    return -math.log(1.0 - random.random()) / lam


def cauchy(x0=0.0, s=1.0):
    return x0 + s * math.tan(math.pi * (random.random() - 0.5))


def Phi(x):
    return 0.5 * (1 + math.erf(x / math.sqrt(2)))

## 1. Zamítací metoda s obdélníkem

### Myšlenka

Vezmeme libovolnou (třeba nenormovanou) funkci $f \ge 0$ na $[a, b]$ a obdélník $[a, b]\times[0, C]$ takový, že $f(x) \le C$ všude. Body vybíráme **rovnoměrně v obdélníku** a ponecháme jen ty, které leží **pod grafem** $f$. Přijaté body jsou rovnoměrně rozložené pod grafem, a jejich souřadnice $x$ má proto hustotu úměrnou $f$.

**Algoritmus:**

1. $\xi_1 \sim R(a, b)$, $\xi_2 \sim R(0, C)$ nezávisle.
2. Je-li $\xi_2 < f(\xi_1)$, vrať $\xi_1$. Jinak zahoď oba a vrať se na krok 1.

### Proč to funguje

Pravděpodobnost, že navržené $\xi_1 \in dx$ je přijato, je $\dfrac{dx}{b - a}\cdot\dfrac{f(x)}{C}$, tedy úměrná $f(x)\,dx$. Po přijetí tedy $\xi_1$ má hustotu $f(x)/\int f$.

Pravděpodobnost přijetí je poměr ploch, $\dfrac{\int f}{C\,(b-a)}$, takže **očekávaný počet pokusů** na jeden vzorek je $\dfrac{C\,(b-a)}{\int f}$ (u normované $f$ jen $C\,(b-a)$).

### Co je vlastně „obdélník" jako funkce

Obdélník je **konstantní funkce** $h(x) = C$ na $[a, b]$: leží nad grafem $f$ a umíme z ní generovat triviálně (rovnoměrně). Pro pevné $x_0$ je pravděpodobnost přijetí poměr dvou délek:

$$P(\text{přijmout} \mid x_0) = \frac{f(x_0)}{C}.$$

Obrázek ukazuje obě veličiny: vlevo geometrii (zelená plocha = přijmeme, červená = zamítneme), vpravo samotnou pravděpodobnost přijetí jako funkci $x$.

In [ ]:
fb = lambda x: math.exp(-x * x)   # naše f (nenormovaná)
a_, b_, C_ = 0.0, 1.0, 1.0        # obdélník [a, b] x [0, C]
grid = [i / 200 for i in range(201)]
x0 = 0.7

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
ax = axes[0]
ax.fill_between(grid, [fb(x) for x in grid], color="tab:green", alpha=0.35, label="pod grafem: přijmeme")
ax.fill_between(grid, [fb(x) for x in grid], [C_] * len(grid), color="tab:red", alpha=0.25, label="nad grafem: zamítneme")
ax.plot(grid, [fb(x) for x in grid], "k", lw=2, label="$f(x) = e^{-x^2}$ (cíl)")
ax.plot([a_, b_], [C_, C_], "b--", lw=2, label="$h(x) = C = 1$ (obdélník, umíme generovat)")
ax.plot([x0, x0], [0, C_], color="gray", ls=":")
ax.annotate("", xy=(x0, fb(x0)), xytext=(x0, 0), arrowprops=dict(arrowstyle="<->", color="tab:green"))
ax.annotate("", xy=(x0, C_), xytext=(x0, fb(x0)), arrowprops=dict(arrowstyle="<->", color="tab:red"))
ax.text(x0 + 0.02, fb(x0) / 2, "$f(x_0)$", color="tab:green")
ax.text(x0 + 0.02, (fb(x0) + C_) / 2, "$C - f(x_0)$", color="tab:red")
ax.set_xlabel("$x$")
ax.set_ylim(0, 1.12)
ax.set_title("Obdélník $[a,b]\\times[0,C]$ a graf $f$")
ax.legend(loc="lower left", fontsize=8)

axes[1].plot(grid, [fb(x) / C_ for x in grid], "k", lw=2)
axes[1].fill_between(grid, [fb(x) / C_ for x in grid], color="tab:green", alpha=0.35)
axes[1].plot([x0, x0], [0, fb(x0) / C_], color="gray", ls=":")
axes[1].scatter([x0], [fb(x0) / C_], color="tab:green", zorder=3)
axes[1].set_ylim(0, 1.12)
axes[1].set_xlabel("$x$")
axes[1].set_title("Pravděpodobnost přijetí $f(x)/C$ v bodě $x$")
plt.tight_layout()
plt.show()

In [ ]:
def zamitaci_krabice(f, C, a, b):
    '''Zamítací metoda s obdélníkem [a, b] x [0, C]. Vrací (x, počet pokusů).'''
    pokusy = 0
    while True:
        pokusy += 1
        x = uniform(a, b)
        y = uniform(0, C)
        if y < f(x):
            return x, pokusy


n = 200_000

# hustota f(x) = 2x na [0, 1], C = 2: E X = 2/3, očekávaný počet pokusů = C (b - a) / 1 = 2
f1 = lambda x: 2 * x
check("f = 2x: E X", *estimate(lambda: zamitaci_krabice(f1, 2, 0, 1)[0], n), 2 / 3)
check("f = 2x: pokusů na vzorek", *estimate(lambda: zamitaci_krabice(f1, 2, 0, 1)[1], n), 2.0)

### Nenormovaná hustota

Teď funkce, kterou jsme integrovali ve cvičení 1: $f(x) = e^{-x^2}$ na $[0, 1]$. Její integrál je $Z = \tfrac{\sqrt\pi}{2}\operatorname{erf}(1) \approx 0.7468$, takže $f$ **není normovaná**, a zamítací metodě to nevadí. Hustota je $f/Z$, její střední hodnota $\dfrac{1}{Z}\displaystyle\int_0^1 x e^{-x^2}\,dx = \dfrac{1 - e^{-1}}{2Z}$ a očekávaný počet pokusů při $C = 1$ je $1/Z$.

In [ ]:
f2 = lambda x: math.exp(-x * x)
Z = math.sqrt(math.pi) / 2 * math.erf(1)

check("exp(-x^2): E X", *estimate(lambda: zamitaci_krabice(f2, 1, 0, 1)[0], n), (1 - math.exp(-1)) / (2 * Z))
check("exp(-x^2): pokusů na vzorek", *estimate(lambda: zamitaci_krabice(f2, 1, 0, 1)[1], n), 1 / Z)

# podíl přijatých bodů krát obsah obdélníku je odhad integrálu (trefa/netrefa ze cvičení 1)
check("plocha pod exp(-x^2) z podílu přijatých", *estimate(lambda: float(uniform(0, 1) < f2(uniform(0, 1))), n), Z)

Poslední řádek je trefa/netrefa z cvičení 1: stejné body, jen jsme je tam **počítali**, a tady je **ponecháváme**.

### Obrázek: simulované body a výsledek

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

prijato, zamitnuto = [], []
for _ in range(1500):
    x, y = uniform(0, 1), uniform(0, 1)
    (prijato if y < f2(x) else zamitnuto).append((x, y))

ax = axes[0]
ax.scatter(*zip(*zamitnuto), s=6, c="tab:red", alpha=0.6, label="zamítnuto")
ax.scatter(*zip(*prijato), s=6, c="tab:green", alpha=0.6, label="přijato")
grid = [i / 200 for i in range(201)]
ax.plot(grid, [f2(x) for x in grid], "k", label="$f(x) = e^{-x^2}$")
ax.set_title(f"Návrhy v obdélníku [0, 1] x [0, 1]: přijato {len(prijato) / 1500:.0%}")
ax.legend(loc="lower left")

xs = [zamitaci_krabice(f2, 1, 0, 1)[0] for _ in range(100_000)]
w = 1 / 40
axes[1].hist(xs, bins=40, range=(0, 1), weights=[1 / (len(xs) * w)] * len(xs), alpha=0.6, label="přijaté $x$")
axes[1].plot(grid, [f2(x) / Z for x in grid], "r", label="hustota $f(x)/Z$")
axes[1].set_title("Přijaté hodnoty mají hustotu úměrnou f")
axes[1].legend()
plt.tight_layout()
plt.show()

Vlevo je vidět celý princip: zelené body (pod křivkou) zůstávají, červené se zahazují. Vpravo je histogram přijatých $x$-ových souřadnic a přesná hustota.

## 2. Obálka místo obdélníku

Obdélník má dvě slabiny: potřebuje **konečný interval** a **omezenou** $f$, a pokud je $f$ „špičatá", obdélník je z větší části prázdný (hodně zamítnutí). Zobecnění: místo obdélníku použijeme **obálku** $M g(x)$, kde $g$ je hustota, ze které umíme generovat, a $M$ je konstanta taková, že

$$f(x) \le M\,g(x) \qquad\text{pro všechna } x.$$

**Algoritmus:**

1. $x \sim g$, $\gamma \sim R(0, 1)$ nezávisle.
2. Je-li $\gamma\,M\,g(x) \le f(x)$, vrať $x$. Jinak se vrať na krok 1.

(Ekvivalentně: přijmi $x$ s pravděpodobností $\dfrac{f(x)}{M g(x)} \le 1$.) Obdélník je speciální případ: $g$ rovnoměrná na $[a, b]$, $M g = C$.

**Očekávaný počet pokusů** je $M$ (jsou-li $f$ i $g$ normované hustoty). Čím těsněji obálka přiléhá, tím je $M$ blíž jedné a tím méně zamítáme.

### Obrázek: graf $f$, obálka $M g$ a pravděpodobnost přijetí

Rozdíl proti obdélníku je jediný: místo konstanty $C$ je horní hranicí křivka $M g(x)$. Pro pevné $x_0$ platí zase

$$P(\text{přijmout} \mid x_0) = \frac{f(x_0)}{M\,g(x_0)}.$$

Příklad: cílová $f$ je standardní normální hustota a $g$ Cauchyho hustota (konstantu $M$ odvodíme v další sekci, tady ji jen použijeme).

In [ ]:
phi_ = lambda x: math.exp(-x * x / 2) / math.sqrt(2 * math.pi)       # cíl f
gc_ = lambda x: 1.0 / (math.pi * (1 + x * x))                         # návrh g (Cauchy)
Mc_ = math.sqrt(math.pi / 2) * 2 * math.exp(-0.5)                     # M
grid = [i / 100 - 5 for i in range(1001)]
x0 = 2.0

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
ax = axes[0]
ax.fill_between(grid, [phi_(x) for x in grid], color="tab:green", alpha=0.35, label="pod $f$: přijmeme")
ax.fill_between(grid, [phi_(x) for x in grid], [Mc_ * gc_(x) for x in grid], color="tab:red", alpha=0.25, label="mezi $f$ a $Mg$: zamítneme")
ax.plot(grid, [phi_(x) for x in grid], "k", lw=2, label="$f(x)$ (cíl, normální hustota)")
ax.plot(grid, [Mc_ * gc_(x) for x in grid], "b--", lw=2, label="$M g(x)$ (obálka, Cauchy)")
ax.plot([x0, x0], [0, Mc_ * gc_(x0)], color="gray", ls=":")
ax.annotate("", xy=(x0, phi_(x0)), xytext=(x0, 0), arrowprops=dict(arrowstyle="<->", color="tab:green"))
ax.annotate("", xy=(x0, Mc_ * gc_(x0)), xytext=(x0, phi_(x0)), arrowprops=dict(arrowstyle="<->", color="tab:red"))
ax.annotate("$f(x_0)$", xy=(x0, phi_(x0) / 2), xytext=(x0 + 1.1, 0.055), color="tab:green", fontsize=9,
            arrowprops=dict(arrowstyle="-", color="tab:green", lw=0.8))
ax.text(x0 + 0.15, Mc_ * gc_(x0) + 0.02, "$Mg(x_0) - f(x_0)$", color="tab:red", fontsize=9)
ax.set_xlabel("$x$")
ax.set_title("Obálka $M g$ nad grafem $f$")
ax.legend(fontsize=8)

axes[1].plot(grid, [phi_(x) / (Mc_ * gc_(x)) for x in grid], "k", lw=2)
axes[1].fill_between(grid, [phi_(x) / (Mc_ * gc_(x)) for x in grid], color="tab:green", alpha=0.35)
axes[1].plot([x0, x0], [0, phi_(x0) / (Mc_ * gc_(x0))], color="gray", ls=":")
axes[1].scatter([x0], [phi_(x0) / (Mc_ * gc_(x0))], color="tab:green", zorder=3)
axes[1].axhline(1, color="b", ls="--", lw=1)
axes[1].set_ylim(0, 1.12)
axes[1].set_xlabel("$x$")
axes[1].set_title("Pravděpodobnost přijetí $f(x)/(M g(x))$ (nikdy nad $1$)")
plt.tight_layout()
plt.show()

In [ ]:
def zamitaci_obalka(f, g_pdf, g_sampler, M):
    '''Zamítací metoda s obálkou M * g >= f. Vrací (x, počet pokusů).'''
    pokusy = 0
    while True:
        pokusy += 1
        x = g_sampler()
        if random.random() * M * g_pdf(x) <= f(x):
            return x, pokusy

## 3. Normální rozdělení z Cauchyho obálky

Cílem je standardní normální hustota $f(x) = \dfrac{1}{\sqrt{2\pi}}e^{-x^2/2}$. Jako $g$ vezmeme Cauchyho hustotu $g(x) = \dfrac{1}{\pi(1 + x^2)}$: umíme z ní generovat inverzní transformací (cvičení 4) a má **těžší chvosty** než $f$, takže obálka nemá problém s nekonečným intervalem.

Konstanta $M$ je supremum poměru:

$$\frac{f(x)}{g(x)} = \sqrt{\frac{\pi}{2}}\,(1 + x^2)\,e^{-x^2/2}.$$

Derivace je nulová pro $x = 0$ a $x^2 = 1$; maximum je v $x = \pm 1$, tedy

$$M = \sup_x \frac{f(x)}{g(x)} = \sqrt{\frac{\pi}{2}}\cdot 2\,e^{-1/2} \approx 1.520.$$

In [ ]:
f = lambda x: math.exp(-x * x / 2) / math.sqrt(2 * math.pi)
g = lambda x: 1.0 / (math.pi * (1 + x * x))
M = math.sqrt(math.pi / 2) * 2 * math.exp(-0.5)
print(f"M = {M:.4f}")


def normalni_cauchy():
    '''Standardní normální rozdělení zamítací metodou s Cauchyho obálkou. Vrací (x, počet pokusů).'''
    return zamitaci_obalka(f, g, cauchy, M)


check("E X", *estimate(lambda: normalni_cauchy()[0], n), 0.0)
check("E X^2", *estimate(lambda: normalni_cauchy()[0] ** 2, n), 1.0)
check("P(X < 1)", *estimate(lambda: float(normalni_cauchy()[0] < 1), n), Phi(1))
check("pokusů na vzorek", *estimate(lambda: normalni_cauchy()[1], n), M)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

# vlevo: návrhy x ~ Cauchy, y ~ R(0, M g(x)); zobrazíme ty s |x| < 5
prijato, zamitnuto = [], []
while len(prijato) + len(zamitnuto) < 2000:
    x = cauchy()
    if abs(x) > 5:
        continue
    y = uniform(0, M * g(x))
    (prijato if y <= f(x) else zamitnuto).append((x, y))

ax = axes[0]
ax.scatter(*zip(*zamitnuto), s=6, c="tab:red", alpha=0.6, label="zamítnuto")
ax.scatter(*zip(*prijato), s=6, c="tab:green", alpha=0.6, label="přijato")
grid = [i / 100 - 5 for i in range(1001)]
ax.plot(grid, [f(x) for x in grid], "k", label="normální hustota $f$")
ax.plot(grid, [M * g(x) for x in grid], "b--", label="obálka $M g$ (Cauchy)")
ax.set_title("Návrhy pod obálkou, přijaté leží pod $f$")
ax.legend()

# vpravo: histogram přijatých hodnot
xs = [normalni_cauchy()[0] for _ in range(200_000)]
w = 8 / 80
axes[1].hist(xs, bins=80, range=(-4, 4), weights=[1 / (len(xs) * w)] * len(xs), alpha=0.6, label="přijaté hodnoty")
axes[1].plot(grid, [f(x) for x in grid], "r", label="$N(0, 1)$")
axes[1].set_xlim(-4, 4)
axes[1].set_title("Zamítací metoda dává normální rozdělení")
axes[1].legend()
plt.tight_layout()
plt.show()

Modrá přerušovaná čára je obálka $M g$: leží všude nad $f$ a dotýká se jí v $x = \pm 1$ (tam je poměr $f/g$ největší). Přijímáme jen návrhy pod křivkou $f$. Plocha pod obálkou je $M \approx 1.52$ a plocha pod $f$ je $1$, takže přijmeme asi $1/1.52 \approx 66\,\%$ návrhů.

## 4. Volba obálky

### Najít $M$

$M$ je supremum $f/g$. Lze ho odvodit **analyticky** (jako výše), nebo hrubě odhadnout numericky na mřížce:

In [ ]:
def najdi_M(f, g, lo=-10.0, hi=10.0, kroku=20_000):
    '''Hrubý numerický odhad sup f/g na mřížce. Přesnější a bezpečnější je odvodit M analyticky.'''
    return max(f(lo + (hi - lo) * i / kroku) / g(lo + (hi - lo) * i / kroku) for i in range(kroku + 1))


print(f"Cauchyho obálka:  numericky {najdi_M(f, g):.4f},  analyticky {M:.4f}")

# Laplaceova (dvojitá exponenciální) obálka: g(x) = e^{-|x|} / 2, generátor = náhodné znaménko * Exp(1)
g_laplace = lambda x: 0.5 * math.exp(-abs(x))
laplace = lambda: exponential(1.0) if random.random() < 0.5 else -exponential(1.0)
M_laplace = math.sqrt(2 * math.e / math.pi)
print(f"Laplaceova obálka: numericky {najdi_M(f, g_laplace):.4f},  analyticky {M_laplace:.4f}")

Numerický odhad se může **mýlit**, pokud je maximum mimo mřížku nebo ostré. Používejte ho k ověření odvozené hodnoty, ne místo odvození.

### Která obálka je lepší?

Menší $M$ znamená méně zamítnutí. Porovnáme Cauchyho a Laplaceovu obálku pro stejnou cílovou hustotu (a k tomu numerickou inverzi půlením z cvičení 4 a knihovní generátor):

In [ ]:
def inverze_pulenim(F, gamma, lo=-10.0, hi=10.0, iteraci=60):
    for _ in range(iteraci):
        mid = (lo + hi) / 2
        if F(mid) < gamma:
            lo = mid
        else:
            hi = mid
    return (lo + hi) / 2


generatory = {
    "Cauchyho obálka": lambda: zamitaci_obalka(f, g, cauchy, M)[0],
    "Laplaceova obálka": lambda: zamitaci_obalka(f, g_laplace, laplace, M_laplace)[0],
    "numerická inverze (cvičení 4)": lambda: inverze_pulenim(Phi, random.random()),
    "random.gauss (knihovna)": lambda: random.gauss(0, 1),
}

m_cas = 50_000
print(f"{'metoda':32s} {'E X^2':>8s} {'čas na losování':>17s}")
for nazev, gen in generatory.items():
    t0 = time.perf_counter()
    m, se = estimate(lambda: gen() ** 2, m_cas)
    t = (time.perf_counter() - t0) / m_cas
    print(f"{nazev:32s} {m:8.4f} {t * 1e6:14.2f} us")

for nazev, gen, MM in (("Cauchy", cauchy, M), ("Laplace", laplace, M_laplace)):
    pg = g if nazev == "Cauchy" else g_laplace
    m, se = estimate(lambda: zamitaci_obalka(f, pg, gen, MM)[1], 100_000)
    check(f"pokusů na vzorek, {nazev} (M = {MM:.4f})", m, se, MM)

Laplaceova obálka je těsnější ($M \approx 1.32$ proti $1.52$), tedy méně zamítnutí. Zamítací metoda je asi $8\times$ rychlejší než numerická inverze a jen asi $1.3$ až $2\times$ pomalejší než knihovní `random.gauss` (ten používá jiný, hodně optimalizovaný algoritmus).

### Obálka musí být nad $f$ všude

Pokud $M g$ někde pod $f$ vede, algoritmus **tiše vrací špatné rozdělení**: v místech, kde $f > M g$, přijímáme s pravděpodobností $1$ místo správné vyšší, takže tam je hustota podhodnocená. Zkusme to s příliš malou konstantou $M$:

In [ ]:
for M_spatne in (M, 1.2, 1.0):
    try:
        check(f"M = {M_spatne:.4f}: E X^2", *estimate(lambda: zamitaci_obalka(f, g, cauchy, M_spatne)[0] ** 2, 300_000), 1.0)
        m, se = estimate(lambda: zamitaci_obalka(f, g, cauchy, M_spatne)[1], 100_000)
        print(f"   pokusů na vzorek: {m:.4f}  (teorie: M = {M_spatne:.4f})")
    except AssertionError:
        m, se = estimate(lambda: zamitaci_obalka(f, g, cauchy, M_spatne)[1], 100_000)
        print(f"   test SELHAL; pokusů na vzorek: {m:.4f}, což je ale víc než M = {M_spatne:.4f}")

S příliš malým $M$ **selže test střední hodnoty $E X^2$** (vyjde víc než $1$). Je tu i druhé varování: průměrný počet pokusů **nesedí** s $M$ (u $M = 1.2$ vyjde kolem $1.36$ místo $1.2$). Přijatá část obálky totiž má obsah menší než $1$, takže na jeden vzorek potřebujeme víc pokusů, než říká teorie. Neodpovídá-li v implementaci změřený počet pokusů hodnotě $M$, hledejte chybu v $M$.

### Když obálka neexistuje

Aby $f \le Mg$ platilo s konečným $M$, musí mít $g$ **chvosty aspoň tak těžké** jako $f$. Zkusme opačnou úlohu: cílem je Cauchyho hustota a obálkou normální. Poměr $f/g$ roste přes všechny meze:

In [ ]:
f_cauchy = lambda x: 1.0 / (math.pi * (1 + x * x))
g_normal = lambda x: math.exp(-x * x / 2) / math.sqrt(2 * math.pi)

print(f"{'x':>3s} {'f_Cauchy(x) / g_normal(x)':>28s}")
for x in (1, 2, 3, 5, 10):
    print(f"{x:3d} {f_cauchy(x) / g_normal(x):28.4g}")

Žádné konečné $M$ neexistuje, takže normální rozdělení jako obálku pro Cauchyho použít nelze (a v tomto případě se tu hodí inverzní transformace, která funguje bez obálky).

## 5. Alternativa: NumPy

Zamítací metodu lze dělat po **dávkách**: navrhneme celé pole kandidátů, zamítneme maskou a opakujeme, dokud nemáme dost. Pozor: dávku musíme volit větší než potřebujeme, protože přijmeme jen $\approx 1/M$ z nich.

In [ ]:
try:
    import numpy as np

    rng = np.random.default_rng(1)

    def normalni_numpy(N):
        '''Zamítací metoda po dávkách: kandidáti z Cauchyho, maska f(x) >= γ M g(x).'''
        vysledek = np.empty(0)
        while vysledek.size < N:
            kolik = int((N - vysledek.size) * M * 1.1) + 100
            x = rng.standard_cauchy(kolik)
            u = rng.random(kolik)
            fx = np.exp(-x * x / 2) / np.sqrt(2 * np.pi)
            gx = 1.0 / (np.pi * (1 + x * x))
            vysledek = np.concatenate([vysledek, x[u * M * gx <= fx]])
        return vysledek[:N]

    N = 1_000_000
    t0 = time.perf_counter()
    x = normalni_numpy(N)
    t_zam = time.perf_counter() - t0

    t0 = time.perf_counter()
    y = rng.standard_normal(N)
    t_lib = time.perf_counter() - t0

    print(f"zamítací metoda: E X = {x.mean():+.4f}, E X^2 = {np.mean(x * x):.4f},  {t_zam * 1e3:.0f} ms")
    print(f"standard_normal: E X = {y.mean():+.4f}, E X^2 = {np.mean(y * y):.4f},  {t_lib * 1e3:.0f} ms")
except ModuleNotFoundError:
    print("NumPy není nainstalované, spusťte:  uv sync")

Zamítací metoda ve vektorové podobě je řádově rychlejší než smyčka v čistém Pythonu, ale knihovní `standard_normal` je pořád rychlejší (používá jiný algoritmus, takzvaný *ziggurat*, což je rafinovaná varianta téže myšlenky s předpočítanými obálkami).

## Shrnutí

| Pojem | Co si pamatovat |
|-------|-----------------|
| Obdélníková verze | body rovnoměrně v $[a,b]\times[0,C]$, ponech ty pod grafem $f$; $f$ nemusí být normovaná |
| Obálková verze | $x \sim g$, přijmi, je-li $\gamma\,M\,g(x) \le f(x)$; vyžaduje $f \le M g$ všude |
| Očekávaný počet pokusů | $M$ (u normovaných hustot); obdélník: $C(b-a)/\int f$ |
| Volba obálky | těsná ($M$ blízko $1$), $g$ musí mít chvosty aspoň tak těžké jako $f$ |
| Špatné $M$ | tichá chyba: rozdělení je zkreslené; odhalí ho test $E X^2$ a počet pokusů, který nesedí s $M$ |
| Normální z Cauchyho | $M = \sqrt{\pi/2}\cdot 2e^{-1/2} \approx 1.52$; z Laplaceovy $M = \sqrt{2e/\pi} \approx 1.32$ |
| Souvislost se cvičením 1 | trefa/netrefa počítá body pod grafem, zamítací metoda je ponechává |

### Zkuste si

1. **Beta(2, 2):** hustota $f(x) = 6x(1-x)$ na $[0, 1]$. Použijte obdélník s nejmenším možným $C$ (maximum je $1.5$ v $x = 1/2$). Ověřte $E X = 1/2$, $D X = 1/20$ a očekávaný počet pokusů $1.5$.
2. **Polonormální rozdělení z exponenciální obálky:** cílová hustota $f(x) = \sqrt{2/\pi}\,e^{-x^2/2}$ pro $x > 0$, obálka $g(x) = e^{-x}$. Odvoďte $M = \sqrt{2e/\pi}$, naimplementujte a ověřte $E X = \sqrt{2/\pi}$. Pak přidejte **náhodné znaménko** a máte $N(0, 1)$ s ještě těsnější obálkou než dříve. Kolik pokusů to stojí?
3. Ve smyčce změňte $M$ ze správné hodnoty o $5\,\%$ dolů. Odhalí to test $E X^2$ při $n = 200\,000$? Kolikrát víc vzorků potřebujete, abyste $2\,\%$ chybu v $M$ poznali?